# 03. Huấn luyện Mô hình & Đánh giá dự phòng

Notebook này thực hiện quá trình huấn luyện và xác thực các lớp mô hình dự báo từ cơ bản đến nâng cao:

1. **Nhóm mô hình cơ sở (Baselines)**:
   - Naive (Dự báo bằng giá trị cuối cùng $y_t$)
   - Moving Average (Trung bình trượt 24 giờ tương ứng cửa sổ 96 bước)
   - Hồi quy tuyến tính Baseline (chỉ sử dụng các biến trễ)

2. **Nhóm mô hình máy học chứa yếu tố mùa vụ (Seasonality / ML)**:
   - Hồi quy tuyến tính kết hợp biến mùa vụ Fourier
   - XGBoost Regressor (Huấn luyện trên tất cả các đặc trưng lịch, Fourier và trễ)

3. **Mô hình học sâu nâng cao (Advanced Model)**:
   - Mạng tuần tự PyTorch LSTM (sử dụng chuỗi lịch sử lookback window dài 24 bước)

In [1]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd
import xgboost as xgb
import torch

from src.models import NaiveModel, MovingAverageModel, train_linear_regression, PyTorchLSTMRegressor

## 1. Tải các tập dữ liệu đã phân tách

In [2]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

print(f"Kích thước Train: {train_df.shape}")
print(f"Kích thước Val: {val_df.shape}")
print(f"Kích thước Test: {test_df.shape}")

Kích thước Train: (48305, 26)
Kích thước Val: (10351, 26)
Kích thước Test: (10352, 26)


## 2. Phân loại đặc trưng và Biến mục tiêu

In [3]:
target_col = 'OT'
raw_features = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL']
calendar_features = ['hour', 'dayofweek', 'month', 'is_weekend']
fourier_features = ['sin_24', 'cos_24', 'sin_168', 'cos_168']
lag_features = [col for col in train_df.columns if 'lag' in col]
rolling_features = [col for col in train_df.columns if 'roll' in col]

y_train = train_df[target_col]
y_val = val_df[target_col]
y_test = test_df[target_col]

## 3. Huấn luyện các mô hình cơ sở (Baselines)

In [4]:
naive_model = NaiveModel()
naive_preds = naive_model.predict(test_df)

ma_model = MovingAverageModel(window=96)
ma_preds = ma_model.predict(test_df)

X_train_lr_base = train_df[lag_features]
X_test_lr_base = test_df[lag_features]
lr_base_model = train_linear_regression(X_train_lr_base, y_train)
lr_base_preds = lr_base_model.predict(X_test_lr_base)

## 4. Huấn luyện mô hình Mùa vụ / Máy học (ML)

In [5]:
fourier_lr_features = lag_features + fourier_features
X_train_lr_fourier = train_df[fourier_lr_features]
X_test_lr_fourier = test_df[fourier_lr_features]
lr_fourier_model = train_linear_regression(X_train_lr_fourier, y_train)
lr_fourier_preds = lr_fourier_model.predict(X_test_lr_fourier)

all_features = raw_features + calendar_features + fourier_features + lag_features + rolling_features
X_train_xgb = train_df[all_features]
X_test_xgb = test_df[all_features]

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train_xgb, y_train)
xgb_preds = xgb_model.predict(X_test_xgb)

## 5. Huấn luyện mô hình nâng cao (PyTorch LSTM)

In [6]:
def create_sequences(df, target_col, feature_cols, seq_len=24):
    X, y = [], []
    feature_data = df[feature_cols].values
    target_data = df[target_col].values
    for i in range(len(df) - seq_len):
        X.append(feature_data[i : i + seq_len])
        y.append(target_data[i + seq_len])
    return np.array(X), np.array(y)

lstm_features = raw_features + calendar_features + fourier_features + rolling_features
seq_len = 24

X_train_seq, y_train_seq = create_sequences(train_df, target_col, lstm_features, seq_len)
X_test_seq, y_test_seq = create_sequences(test_df, target_col, lstm_features, seq_len)

print("Kích thước tensor huấn luyện X:", X_train_seq.shape)
print("Kích thước nhãn huấn luyện y:", y_train_seq.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
lstm_regressor = PyTorchLSTMRegressor(
    input_dim=len(lstm_features),
    hidden_dim=32,
    num_layers=2,
    lr=0.005,
    epochs=5,
    batch_size=128,
    device=device
)
lstm_regressor.fit(X_train_seq, y_train_seq)

lstm_preds = lstm_regressor.predict(X_test_seq)

aligned_test_df = test_df.iloc[seq_len:].reset_index(drop=True)
y_test_aligned = y_test_seq

Kích thước tensor huấn luyện X: (48281, 24, 18)
Kích thước nhãn huấn luyện y: (48281,)


## 6. Lưu trữ kết quả dự báo của tất cả mô hình

In [7]:
predictions_df = pd.DataFrame({
    "date": aligned_test_df["date"],
    "y_true": y_test_aligned,
    "naive": naive_preds[seq_len:],
    "moving_average": ma_preds[seq_len:],
    "linear_regression_baseline": lr_base_preds[seq_len:],
    "linear_regression_fourier": lr_fourier_preds[seq_len:],
    "xgboost": xgb_preds[seq_len:],
    "lstm": lstm_preds
})

os.makedirs('../results', exist_ok=True)
predictions_df.to_csv('../results/predictions.csv', index=False)
print("Lưu tệp predictions.csv thành công.")

Lưu tệp predictions.csv thành công.
